In [1]:
import pandas as pd
import numpy as np
import time
import os

import lightgbm as lgb
from lightgbm import LGBMRanker
import shap

In [2]:
def load_data (split):
    df_train = pd.read_csv(f"./splits/{split}/train_pares.csv")
    df_val = pd.read_csv(f"./splits/{split}/val_pares.csv")
    df_test = pd.read_csv(f"./splits/{split}/test_pares.csv")

    return df_train, df_val, df_test

In [3]:
def ranking_metrics(group):
    group_id = group.name

    group = group.sort_values("score", ascending=False)

    labels = group["label"].values
    relevant_positions = np.where(labels == 1)[0]

    n_relevant = len(relevant_positions)

    #se não há relevantes, estoura erro
    if n_relevant == 0:
        #print(f"Nenhum parente encontrado id={group_id}")
        return pd.Series({
            "id": group_id,
            "recall@1": np.nan,
            "recall@5": np.nan,
            "recall@10": np.nan,
            "recall@15": np.nan,
            "recall@20": np.nan,
            "mrr": np.nan,
            "count": np.nan
        })

    # Recall@1
    recall_at_1 = int(labels[0] == 1) / n_relevant

    # Recall@5
    recall_at_5 = labels[:5].sum() / n_relevant

    # Recall@10
    recall_at_10 = labels[:10].sum() / n_relevant

    # Recall@10
    recall_at_15 = labels[:15].sum() / n_relevant

    # Recall@10
    recall_at_20 = labels[:20].sum() / n_relevant

    # MRR
    first_relevant_rank = relevant_positions[0] + 1
    mrr = 1.0 / first_relevant_rank

    return pd.Series({
        "id": group_id,
        "recall@1": recall_at_1,
        "recall@5": recall_at_5,
        "recall@10": recall_at_10,
        "recall@15": recall_at_15,
        "recall@20": recall_at_20,
        "mrr": mrr,
        "count": len(labels)
    })

In [4]:
#nova célula 4 com função inferencia 
def inferencia(label, df_test, split_path, features, model, mode="ranker"):
    """Realiza inferência e calcula métricas de ranking"""
    
    X_test = df_test[features].copy()
    
    if mode == "heuristic":
        cols_to_fill = ["age_score", "addr_score", "occ_score"]
        X_test[cols_to_fill] = X_test[cols_to_fill].fillna(0)
        scores = (
            0.50 * X_test["age_score"] +
            0.40 * X_test["addr_score"] +
            0.10 * X_test["occ_score"]
        )
    
    elif mode == "classifier":
        cols_to_fill = ["age_score", "addr_score", "occ_score", "addr_exato"]
        X_test[cols_to_fill] = X_test[cols_to_fill].fillna(0)
        scores = model.predict_proba(X_test)[:, 1]
    
    elif mode == "ranker":
        scores = model.predict(X_test)
    
    df_test_copy = df_test.copy()
    df_test_copy["score"] = scores
    
    df_results = df_test_copy.groupby(["ID"]).apply(ranking_metrics)
    df_results = df_results.reset_index(drop=True)
    df_results.to_csv(f"{split_path}/{label}.csv", index=False)
    
    return df_results





In [5]:
def lgbm_explainer (split, df_test, split_path, features, feature_groups, model_lgbm):
    explainer = shap.TreeExplainer(model_lgbm)
    
    X = df_test[features]
    shap_values = explainer.shap_values(X)
    
    importance = np.abs(shap_values).mean(axis=0)
    
    df_importance = pd.DataFrame({
        "feature": X.columns,
        "importance": importance
    }).sort_values("importance", ascending=False)
    
    total = df_importance["importance"].sum()

    group_importance = []
    for group, features in feature_groups.items():
        
        imp = df_importance[
            df_importance["feature"].isin(features)
        ]["importance"].sum()
        
        group_importance.append({
            "group": group,
            "importance": imp
        })
    
    df_group = pd.DataFrame(group_importance)
    
    df_group["relative_importance"] = (
        100 * (df_group["importance"] / total)
    )
    output_file = f"{split_path}/lgbm_explainer.csv"
    df_group.to_csv(output_file, index=False)

In [6]:
#célula nova do lgbm

def train_lgbm_ranker(df_train, df_valid, features):
    """Treina modelo LightGBM Ranker"""
    
    df_train = df_train.sort_values("ID")
    df_valid = df_valid.sort_values("ID")
    
    X_train = df_train[features]
    Y_train = df_train["label"]

    X_valid = df_valid[features]
    Y_valid = df_valid["label"]
    
    #grupos para ranking (precisa ser lista para LightGBM)
    group_train = df_train.groupby("ID").size().tolist()
    group_valid = df_valid.groupby("ID").size().tolist()

    model = LGBMRanker(
        objective="lambdarank",
        metric="ndcg",
        n_estimators=300,
        learning_rate=0.05,
        max_depth=-1,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )
    
    model.fit(
        X_train,
        Y_train,
        group=group_train,
        eval_set=[(X_valid, Y_valid)],
        eval_group=[group_valid],
        eval_at=[1, 5, 10],
        callbacks=[
            lgb.log_evaluation(period=20)
        ]
    )

    return model

In [ ]:
#célula nova com heurística, xgboost e lgbm

OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)


for split in range(1, 6):
    print(f"\n{'='*50}")
    print(f"SPLIT {split}")
    print(f"{'='*50}\n")
    
    split_path = OUTPUT_PATH+f"/{split}"
    os.makedirs(split_path, exist_ok=True)
    
    # Carregar dados
    print(f"Loading Data {split}")
    df_train, df_val, df_test = load_data(split)
    
    # Shuffle dos dados de treino
    print(f"Shuffling train data {split}")
    df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)
    
    # Criar one-hot encoding
    print(f"Creating one-hot encoding {split}")
    df_train = pd.get_dummies(df_train, columns=['RELATED'], prefix='is')
    df_val = pd.get_dummies(df_val, columns=['RELATED'], prefix='is')
    df_test = pd.get_dummies(df_test, columns=['RELATED'], prefix='is')
    
    # Garantir que todos os dataframes tenham as mesmas colunas
    all_columns = list(set(df_train.columns) | set(df_val.columns) | set(df_test.columns))
    for col in all_columns:
        if col not in df_train.columns:
            df_train[col] = 0
        if col not in df_val.columns:
            df_val[col] = 0
        if col not in df_test.columns:
            df_test[col] = 0
    
    # Atualizar features com as novas colunas is_*
    is_columns = [col for col in df_train.columns if col.startswith('is_')]
    features_rank = ['age_score', 'addr_score', 'addr_exato', 'occ_score'] + is_columns

    feature_groups = {
        "address": ["addr_score", "addr_exato"],
        "age": ["age_score"],
        "profession": ["occ_score"],
        "relationship_indicators": is_columns
    }  
    # ==================== LightGBM Ranker ====================
    label = "lgbm_ranker"
    print("Training model LGBM", split)
    lgbm_model = train_lgbm_ranker(df_train, df_val, features_rank)
    print(f"Inference LightGBM Ranker {split}")
    inferencia(label, df_test, split_path, features_rank, lgbm_model, mode="ranker")
    print("LGBM Explainer", split)
    lgbm_explainer (split, df_test, split_path, features_rank, feature_groups, lgbm_model)

print("fim")


SPLIT 1

Loading Data 1
Shuffling train data 1
Creating one-hot encoding 1
Training model LGBM 1
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.103131 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 793
[LightGBM] [Info] Number of data points in the train set: 4758531, number of used features: 17
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[20]	valid_0's ndcg@1: 0.95087	valid_0's ndcg@5: 0.979057	valid_0's ndcg@10: 0.979657
[40]	valid_0's ndcg@1: 0.951237	valid_0's ndcg@5: 0.979206	valid_0's ndcg@10: 0.979814
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[60]	valid_0's ndcg@1: 0.951364	valid_0's ndcg@5: 0.979219	valid_0's ndcg@10: 0.979832
[80]	valid_0's ndcg@1: 0.951779	valid_0's ndcg@5: 0.979355	valid_0's ndcg@10: 0.979973
[100]	valid_0's ndcg@1: 0.9518

In [11]:
import pandas as pd
import numpy as np
import os

grupos = ["address", "age", "profession", "relationship_indicators", "name_mother_father"]
resultados = []

for split in [1, 2, 3, 4, 5]:    caminho = f"./output/{split}/lgbm_explainer.csv" 
    
    linha = {"Split": str(split)}
    
    if os.path.exists(caminho):
        df_csv = pd.read_csv(caminho)
        # Mapeia os grupos com suas respectivas importâncias relativas
        for _, row in df_csv.iterrows():
            if row["group"] in grupos:
                linha[row["group"]] = row["relative_importance"]
                
    resultados.append(linha)

# Cria o DataFrame base
df_tabela = pd.DataFrame(resultados)

for col in grupos:
    if col not in df_tabela.columns:
        df_tabela[col] = np.nan

# Ordena as colunas
df_tabela = df_tabela[["Split"] + grupos]

# Converte para numérico para cálculos
df_tabela[grupos] = df_tabela[grupos].astype(float)

# Calcula Média e Desvio Padrão
media = df_tabela[grupos].mean().to_dict()
media["Split"] = "Media"

desvio = df_tabela[grupos].std(ddof=0).to_dict()
desvio["Split"] = "Desvio"

df_resumo = pd.DataFrame([media, desvio])
df_final = pd.concat([df_tabela, df_resumo], ignore_index=True)

df_final[grupos] = df_final[grupos].round(4)

display(df_final)

,Split,address,age,profession,relationship_indicators,name_mother_father
0,1,60.7055,19.2988,18.7269,1.2688,NaN
1,2,58.6978,19.1446,21.2928,0.8648,NaN
2,3,59.4688,17.4857,21.3919,1.6536,NaN
3,4,58.5953,19.5221,20.9853,0.8973,NaN
4,5,58.5519,19.0253,21.5113,0.9115,NaN
5,Media,59.2039,18.8953,20.7816,1.1192,NaN
6,Desvio,0.8218,0.7241,1.0421,0.3050,NaN
